In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download()

NLTK Downloader
---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> d

Download which package (l=list; x=cancel)?
  Identifier> all


       | 
       | Downloading package abc to /root/nltk_data...
       |   Unzipping corpora/abc.zip.
       | Downloading package alpino to /root/nltk_data...
       |   Unzipping corpora/alpino.zip.
       | Downloading package averaged_perceptron_tagger to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger.zip.
       | Downloading package averaged_perceptron_tagger_eng to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
       | Downloading package averaged_perceptron_tagger_ru to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_ru.zip.
       | Downloading package averaged_perceptron_tagger_rus to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_rus.zip.
       | Downloading package basque_grammars to /root/nltk_data...
       |   Unzipping grammars/basque_grammars.zip.
       | Downloading package bcp47 to /root/nltk_d


---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> q


True

In [9]:
import pandas as pd
import re
import numpy as np
from collections import Counter
from nltk.tokenize import regexp_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import matplotlib.pyplot as plt


df_train = pd.read_csv('Corona_NLP_train.csv', encoding='latin1')
df_test = pd.read_csv('Corona_NLP_test.csv', encoding='latin1')
# deo pod a)
stop_words = set(stopwords.words('english'))
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def clean_tweet(tweet):
    tweet = str(tweet).lower()
    tweet = re.sub(r'http\S+', '', tweet)
    tweet = re.sub(r'@\w+', '', tweet)
    tweet = re.sub(r'#', '', tweet)
    tweet = re.sub(r'[^a-z\s]', '', tweet)
    tweet = re.sub(r'\s+', ' ', tweet).strip()
    tokens = regexp_tokenize(tweet, r"[\w']+")
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [porter.stem(w) for w in tokens]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return tokens

df_train['CleanTweet'] = df_train['OriginalTweet'].apply(clean_tweet)
df_test['CleanTweet'] = df_test['OriginalTweet'].apply(clean_tweet)

def simplify_sentiment(s):
    if s in ['Extremely Positive', 'Positive']:
        return 'Positive'
    elif s in ['Extremely Negative', 'Negative']:
        return 'Negative'
    else:
        return 'Neutral'

df_train['SentimentSimple'] = df_train['Sentiment'].apply(simplify_sentiment)
df_test['SentimentSimple'] = df_test['Sentiment'].apply(simplify_sentiment)

df_train = df_train[df_train['SentimentSimple'] != 'Neutral'].reset_index(drop=True)
df_test = df_test[df_test['SentimentSimple'] != 'Neutral'].reset_index(drop=True)

vocab_counter = Counter()
for tokens in df_train['CleanTweet']:
    vocab_counter.update(tokens)
most_common_words = [w for w, _ in vocab_counter.most_common(10000)]
vocab = {word: idx for idx, word in enumerate(most_common_words)}

def bow_vector(tokens, vocab):
    vec = np.zeros(len(vocab), dtype=np.float32)
    for t in tokens:
        if t in vocab:
            vec[vocab[t]] += 1
    return vec

x_train = np.array([bow_vector(tokens, vocab) for tokens in df_train['CleanTweet']])
x_test = np.array([bow_vector(tokens, vocab) for tokens in df_test['CleanTweet']])

label_map = {'Positive': 1, 'Negative': 0}
y_train = np.array([label_map[l] for l in df_train['SentimentSimple']])
y_test = np.array([label_map[l] for l in df_test['SentimentSimple']])

class MultinomialNaiveBayes:
    def __init__(self, nb_classes, nb_words, pseudocount=1):
        self.nb_classes = nb_classes
        self.nb_words = nb_words
        self.pseudocount = pseudocount

    def fit(self, X, Y):
        nb_examples = X.shape[0]

        self.priors = np.bincount(Y) / nb_examples

        occs = np.zeros((self.nb_classes, self.nb_words))
        for i in range(nb_examples):
            c = Y[i]
            for w in range(self.nb_words):
                occs[c][w] += X[i][w]

        self.like = np.zeros((self.nb_classes, self.nb_words))
        for c in range(self.nb_classes):
            total_words = np.sum(occs[c])
            for w in range(self.nb_words):
                up = occs[c][w] + self.pseudocount
                down = total_words + self.nb_words * self.pseudocount
                self.like[c][w] = up / down

    def predict_one(self, bow):
        probs = np.zeros(self.nb_classes)

        for c in range(self.nb_classes):
            prob = np.log(self.priors[c])
            for w in range(self.nb_words):
                cnt = bow[w]
                prob += cnt * np.log(self.like[c][w])
            probs[c] = prob

        return np.argmax(probs)

    def predict(self, X):
        predictions = []
        for i in range(X.shape[0]):
            pred = self.predict_one(X[i])
            predictions.append(pred)
        return np.array(predictions)

nb_model = MultinomialNaiveBayes(
    nb_classes=2,
    nb_words=x_train.shape[1],
    pseudocount=1
)

nb_model.fit(x_train, y_train)

Y_pred = nb_model.predict(x_test)
accuracy = (Y_pred == y_test).mean()

print("A deo")
print("Accuracy na test skupu:", accuracy)


# deo pod b)
print("Deo pod B")
pos_tweets = df_train[df_train['SentimentSimple'] == 'Positive']['CleanTweet']
neg_tweets = df_train[df_train['SentimentSimple'] == 'Negative']['CleanTweet']

pos_counter = Counter()
for tokens in pos_tweets:
  pos_counter.update(tokens)

neg_counter = Counter()
for tokens in neg_tweets:
  neg_counter.update(tokens)

top5_pos = pos_counter.most_common(5)
print("Top 5 najcescih reci u pozitivnim tweets", top5_pos)

top5_neg = neg_counter.most_common(5)
print("Top 5 negativnih reci u negativnim tweers", top5_neg)

common_words = set([w for w, c in pos_counter.items() if c >= 10]) & set([w for w, c in neg_counter.items() if c >= 10])

lr_dict = {}
for w in common_words:
  lr_dict[w] = pos_counter[w] / neg_counter[w]

top5_lr_max = sorted(lr_dict.items(), key=lambda x: x[1], reverse=True)[:5]
print("5 reci sa najvecim LR vrednoscu:", top5_lr_max)

top5_lr_min = sorted(lr_dict.items(), key=lambda x: x[1])[:5]
print("5 reči sa najmanjom LR vrednošću:", top5_lr_min)

A deo
Accuracy na test skupu: 0.7873545139981126
Deo pod B
Top 5 najcescih reci u pozitivnim tweets [('covid', 9611), ('coronaviru', 7501), ('store', 3906), ('price', 3338), ('supermarket', 3282)]
Top 5 negativnih reci u negativnim tweers [('covid', 8021), ('coronaviru', 6719), ('price', 4344), ('food', 3638), ('supermarket', 3009)]
5 reci sa najvecim LR vrednoscu: [('best', 15.448275862068966), ('happi', 11.285714285714286), ('love', 10.676470588235293), ('amaz', 10.285714285714286), ('free', 10.166666666666666)]
5 reči sa najmanjom LR vrednošću: [('crude', 0.06976744186046512), ('scam', 0.1050656660412758), ('worst', 0.11504424778761062), ('fail', 0.11764705882352941), ('war', 0.11790393013100436)]


# komentar za deo b
Top 5 najčešće korišćenih reči u pozitivnim tvitovima pokazuju koje teme se najčešće pominju, npr. 'covid', 'coronaviru', 'store', 'price', 'supermarket'.

Top 5 negativnih reči pokazuju koje reči dominiraju u negativnim tvitovima, npr. 'covid', 'coronaviru', 'price', 'food', 'supermarket'.

Metrika LR (Likelihood Ratio) kvantifikuje koliko je neka reč karakteristična za pozitivne ili negativne tvitove.

Reči sa LR > 1 (najveći LR) su tipične za pozitivne tvitove: 'best', 'happi', 'love', 'amaz', 'free'.

Reči sa LR < 1 (najmanji LR) su tipične za negativne tvitove: 'crude', 'scam', 'worst', 'fail', 'war'.

Ova analiza potvrđuje da LR metrika pomaže identifikaciji ključnih reči koje nose sentiment, a filtriranje reči koje se pojavljuju barem 10 puta u oba korpusa čini metriku pouzdanim.